<a href="https://colab.research.google.com/github/Thampi-hub/Springboard_RT/blob/main/Capstone_2/2_DataWrangling/readData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
#Read in JSON files (Category names):
# ca_json = pd.read_json("./Kaggle_data/CA_category_id.json")["items"].to_dict()
# us_json = pd.read_json("./Kaggle_data/US_category_id.json")["items"].to_dict()
#       --------- OR ---------
url_ca_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CA_category_id.json"
url_us_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/US_category_id.json"
ca_json = pd.read_json(url_ca_json)["items"].to_dict()
us_json = pd.read_json(url_us_json)["items"].to_dict()

cat_id  = []
cat_val = []
for idx in ca_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])
for idx in us_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])

category_mapping = pd.DataFrame({'category_id':cat_id, 'category':cat_val }).drop_duplicates()

In [3]:
#Read in CSV files (Core data files):
# ca_csv = pd.read_csv("./Kaggle_data/CAvideos.csv")
# us_csv = pd.read_csv("./Kaggle_data/USvideos.csv")
#       --------- OR ---------
url_ca_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CAvideos.csv"
url_us_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/USvideos.csv"
ca_csv = pd.read_csv(url_ca_csv)
us_csv = pd.read_csv(url_us_csv)

ca_csv['country'] = "Canada"
us_csv['country'] = "USA"

allCSV = pd.concat([ca_csv,us_csv], axis=0)
print("CSV dimensions: ", ca_csv.shape, " + ", us_csv.shape, " = ", allCSV.shape)


CSV dimensions:  (40881, 17)  +  (40949, 17)  =  (81830, 17)


In [4]:
#Merge all data:
allData = pd.merge(allCSV, category_mapping, on="category_id", how="left")

print("Dimensions: ", allData.shape, "\n\n")
print(allData.columns.values,"\n")
print(allData.dtypes)

Dimensions:  (81830, 18) 


['video_id' 'trending_date' 'title' 'channel_title' 'category_id'
 'publish_time' 'tags' 'views' 'likes' 'dislikes' 'comment_count'
 'thumbnail_link' 'comments_disabled' 'ratings_disabled'
 'video_error_or_removed' 'description' 'country' 'category'] 

video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
country                   object
category                  object
dtype: object


In [5]:
allData['trending_date'] = pd.to_datetime(allData['trending_date'], format="%y.%d.%m")

allData['publish_datetime']  = pd.to_datetime(allData['publish_time'], utc=True)
allData['publish_date']  = allData['publish_datetime'].dt.date
allData['publish_time']  = allData['publish_datetime'].dt.time


In [19]:
id_cols = ["video_id","publish_datetime","trending_date","title","channel_title","category","country","likes","dislikes","comment_count","views"]
new_col_order = id_cols + allData.columns[ ~allData.columns.isin(id_cols)].tolist()
allData = allData[new_col_order]
allData.head()

KeyError: ('video_id', 'publish_datetime', 'trending_date', 'title', 'channel_title', 'category', 'country', 'likes', 'dislikes', 'comment_count', 'views')

In [84]:
clean_idx = allData[allData.title!="Deleted video"].\
            sort_values(["video_id","country","publish_datetime","trending_date"]).\
            groupby(["video_id","country"])["publish_datetime"].\
            idxmax().values
clData = allData[allData.index.isin(clean_idx)]
#clData.shape
a = allData[["video_id","country"]].drop_duplicates().groupby("video_id").size()
a[a>1]
allData[allData.video_id=="-1Hm41N0dUs"]

,video_id,publish_datetime,trending_date,title,channel_title,category,country,likes,dislikes,comment_count,...,category_id,publish_time,tags,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,publish_date,n_tags
31329,-1Hm41N0dUs,2018-04-27 07:30:02+00:00,2018-04-28,Cast of Avengers: Infinity War Draws Their Cha...,Jimmy Kimmel Live,Comedy,Canada,24616,181,1128,...,23,07:30:02,"jimmy|""jimmy kimmel""|""jimmy kimmel live""|""late...",https://i.ytimg.com/vi/-1Hm41N0dUs/default.jpg,False,False,False,"Benedict Cumberbatch, Don Cheadle, Elizabeth O...",2018-04-27,23
31548,-1Hm41N0dUs,2018-04-27 07:30:02+00:00,2018-04-29,Cast of Avengers: Infinity War Draws Their Cha...,Jimmy Kimmel Live,Comedy,Canada,32753,393,1490,...,23,07:30:02,"jimmy|""jimmy kimmel""|""jimmy kimmel live""|""late...",https://i.ytimg.com/vi/-1Hm41N0dUs/default.jpg,False,False,False,"Benedict Cumberbatch, Don Cheadle, Elizabeth O...",2018-04-27,23
31825,-1Hm41N0dUs,2018-04-27 07:30:02+00:00,2018-04-30,Cast of Avengers: Infinity War Draws Their Cha...,Jimmy Kimmel Live,Comedy,Canada,38165,530,1412,...,23,07:30:02,"jimmy|""jimmy kimmel""|""jimmy kimmel live""|""late...",https://i.ytimg.com/vi/-1Hm41N0dUs/default.jpg,False,False,False,"Benedict Cumberbatch, Don Cheadle, Elizabeth O...",2018-04-27,23
72434,-1Hm41N0dUs,2018-04-27 07:30:02+00:00,2018-04-29,Cast of Avengers: Infinity War Draws Their Cha...,Jimmy Kimmel Live,Comedy,USA,32752,393,1490,...,23,07:30:02,"jimmy|""jimmy kimmel""|""jimmy kimmel live""|""late...",https://i.ytimg.com/vi/-1Hm41N0dUs/default.jpg,False,False,False,"Benedict Cumberbatch, Don Cheadle, Elizabeth O...",2018-04-27,23
72654,-1Hm41N0dUs,2018-04-27 07:30:02+00:00,2018-04-30,Cast of Avengers: Infinity War Draws Their Cha...,Jimmy Kimmel Live,Comedy,USA,38165,530,1412,...,23,07:30:02,"jimmy|""jimmy kimmel""|""jimmy kimmel live""|""late...",https://i.ytimg.com/vi/-1Hm41N0dUs/default.jpg,False,False,False,"Benedict Cumberbatch, Don Cheadle, Elizabeth O...",2018-04-27,23
72873,-1Hm41N0dUs,2018-04-27 07:30:02+00:00,2018-05-01,Cast of Avengers: Infinity War Draws Their Cha...,Jimmy Kimmel Live,Comedy,USA,41248,580,1484,...,23,07:30:02,"jimmy|""jimmy kimmel""|""jimmy kimmel live""|""late...",https://i.ytimg.com/vi/-1Hm41N0dUs/default.jpg,False,False,False,"Benedict Cumberbatch, Don Cheadle, Elizabeth O...",2018-04-27,23


In [37]:
cnt_id   = clData[["video_id","country","publish_datetime"]].drop_duplicates().groupby(["video_id","country"]).size().reset_index().rename(columns={0:'count'})
dupli_id = cnt_id.loc[cnt_id["count"]>1, "video_id"]

clData.sort_values(by=["video_id","country","publish_datetime"])[clData["video_id"].isin(dupli_id)]
# Select max(publish datetime) and then select min(trending date)

<ipython-input-37-48baf82c2bb3>:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clData.sort_values(by=["video_id","country","publish_datetime"])[clData["video_id"].isin(dupli_id)]


,video_id,publish_datetime,trending_date,title,channel_title,category,country,likes,dislikes,comment_count,...,category_id,publish_time,tags,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,publish_date,n_tags
9592,5ajEkB3zzuk,2017-12-31 12:24:52+00:00,2018-01-01,Watch Live: 2018 New Year celebrations around ...,NBC News,News & Politics,Canada,13707,1336,0,...,25,12:24:52,"nbc news|""breaking news""|""us news""|""world news...",https://i.ytimg.com/vi/5ajEkB3zzuk/default_liv...,True,False,False,"From Sydney to New York City, count down to 20...",2017-12-31,27
9795,5ajEkB3zzuk,2018-01-01 09:25:16+00:00,2018-01-02,Watch Live: 2018 New Year celebrations around ...,NBC News,News & Politics,Canada,14958,1697,0,...,25,09:25:16,"nbc news|""breaking news""|""us news""|""world news...",https://i.ytimg.com/vi/5ajEkB3zzuk/default.jpg,True,False,False,"From Sydney to New York City, count down to 20...",2018-01-01,27
10012,5ajEkB3zzuk,2018-01-01 09:25:16+00:00,2018-01-03,Watch Live: 2018 New Year celebrations around ...,NBC News,News & Politics,Canada,15097,1746,0,...,25,09:25:16,"nbc news|""breaking news""|""us news""|""world news...",https://i.ytimg.com/vi/5ajEkB3zzuk/default.jpg,True,False,False,"From Sydney to New York City, count down to 20...",2018-01-01,27
10244,5ajEkB3zzuk,2018-01-01 09:25:16+00:00,2018-01-04,Watch Live: 2018 New Year celebrations around ...,NBC News,News & Politics,Canada,15128,1764,0,...,25,09:25:16,"nbc news|""breaking news""|""us news""|""world news...",https://i.ytimg.com/vi/5ajEkB3zzuk/default.jpg,True,False,False,"From Sydney to New York City, count down to 20...",2018-01-01,27
10560,5ajEkB3zzuk,2018-01-01 09:25:16+00:00,2018-01-05,Watch Live: 2018 New Year celebrations around ...,NBC News,News & Politics,Canada,15135,1770,0,...,25,09:25:16,"nbc news|""breaking news""|""us news""|""world news...",https://i.ytimg.com/vi/5ajEkB3zzuk/default.jpg,True,False,False,"From Sydney to New York City, count down to 20...",2018-01-01,27
18150,FeRi86DcfyA,2018-02-01 03:05:29+00:00,2018-02-15,正在直播：2018中央电视台春节联欢晚会 | 2018 CCTV Spring Festiv...,CCTV春晚,Entertainment,Canada,8034,1197,0,...,24,03:05:29,"春晚2018|""央视春晚""|""春晚""|""春晚直播""|""春节联欢晚会""|""2018春节联欢晚会...",https://i.ytimg.com/vi/FeRi86DcfyA/default_liv...,False,False,False,每年除夕夜，央视春晚都是全球华人最熟悉的陪伴、最热闹的联欢；在海外，看春晚也是与家人共庆佳节...,2018-02-01,14
18348,FeRi86DcfyA,2018-02-15 18:27:21+00:00,2018-02-16,直播回看：2018中央电视台春节联欢晚会 | 2018 CCTV Spring Festiv...,CCTV春晚,Entertainment,Canada,12742,2716,3300,...,24,18:27:21,"央视春晚|""李易峰""|""钟汉良""|""周杰伦""|""言承旭""|""杨洋""|""容祖儿""|""周渝民""|...",https://i.ytimg.com/vi/FeRi86DcfyA/default.jpg,False,False,False,每年除夕夜，央视春晚都是全球华人最熟悉的陪伴、最热闹的联欢；在海外，看春晚也是与家人共庆佳节...,2018-02-15,67
18548,FeRi86DcfyA,2018-02-15 18:27:21+00:00,2018-02-17,直播回看：2018中央电视台春节联欢晚会（完整版） | 2018 CCTV Spring F...,CCTV春晚,Entertainment,Canada,15318,3757,5362,...,24,18:27:21,"央视春晚|""李易峰""|""钟汉良""|""周杰伦""|""言承旭""|""杨洋""|""容祖儿""|""周渝民""|...",https://i.ytimg.com/vi/FeRi86DcfyA/default.jpg,False,False,False,每年除夕夜，央视春晚都是全球华人最熟悉的陪伴、最热闹的联欢；在海外，看春晚也是与家人共庆佳节...,2018-02-15,67
18747,FeRi86DcfyA,2018-02-15 18:27:21+00:00,2018-02-18,直播回看：2018中央电视台春节联欢晚会（完整版） | 2018 CCTV Spring F...,CCTV春晚,Entertainment,Canada,16429,4274,5969,...,24,18:27:21,"央视春晚|""李易峰""|""钟汉良""|""周杰伦""|""言承旭""|""杨洋""|""容祖儿""|""周渝民""|...",https://i.ytimg.com/vi/FeRi86DcfyA/default.jpg,False,False,False,每年除夕夜，央视春晚都是全球华人最熟悉的陪伴、最热闹的联欢；在海外，看春晚也是与家人共庆佳节...,2018-02-15,67
18956,FeRi86DcfyA,2018-02-15 18:27:21+00:00,2018-02-19,直播回看：2018中央电视台春节联欢晚会（完整版） | 2018 CCTV Spring F...,CCTV春晚,Entertainment,Canada,17097,4616,6232,...,24,18:27:21,"央视春晚|""李易峰""|""钟汉良""|""周杰伦""|""言承旭""|""杨洋""|""容祖儿""|""周渝民""|...",https://i.ytimg.com/vi/FeRi86DcfyA/default.jpg,False,False,False,37:24 2018央视春节联欢晚会\n37:30 歌曲《万紫千红中国年》凤凰传奇 容祖儿 ...,2018-02-15,67


In [8]:
# #No. of tags:
# allData['n_tags'] = allData['tags'].apply(lambda x: len(x.split("|")) )

# #No. of Repeats:
# video_n  = allData.groupby('video_id')['video_id'].count()
# vid_n_DF = pd.DataFrame({"video_id" : video_n.index,
#                          "trend_rep": video_counts.values})
# allData = pd.merge(allData, vid_n_DF, on="video_id", how="left")
# allData.shape